# 정형 데이터 EDA

각 Notebook은 독립 실행합니다. 기본값 DEMO=True는 합성 연습 데이터입니다. 실제 데이터는 설정 셀에서 DEMO=False와 경로·열·문제 유형을 지정하세요. 앞의 Notebook 실행이나 개인 모듈 설치가 필요하지 않습니다.

시간 예산은 모델 후보를 시작하기 전에 확인하는 소프트 제한입니다. 진행 중인 fit을 강제 중단하지 않습니다. 대회 지문과 공식 제출 규격을 우선합니다.

## 설정

수치·범주·결측·이상치·불균형·중복을 순서대로 확인합니다. 상관관계는 인과관계가 아닙니다.

In [ ]:
DEMO=True
TASK='classification' # classification / regression
TRAIN_PATH='data/train.csv';TEST_PATH='data/test.csv'
TARGET='target';ID='id'
SPLIT='stratified' # random / stratified / group / time
GROUP=None;TIME_COL=None;GAP=0
DROP_COLUMNS=[] # 사후 정보, 누수 열, 자유 텍스트 등을 제외
METRIC='f1_macro' # 회귀: rmse/mae/r2
ROBUST=False;BUDGET=120
PREDICTION_KIND='label' # label / probability / positive_probability
CLASS_ORDER=None;POSITIVE_CLASS=None
TARGET_COLUMNS=['target'];SAMPLE_PATH=None
OUTPUT='outputs/tabular/submission.csv'


## 공통 함수

데이터 로딩과 유효성 검사

In [ ]:
import os, time, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import (accuracy_score, f1_score, log_loss, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score, classification_report,
    ConfusionMatrixDisplay, silhouette_score)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, IsolationForest
from sklearn.cluster import MiniBatchKMeans
from sklearn.dummy import DummyClassifier, DummyRegressor
SEED=42
rng=np.random.default_rng(SEED)

def read_table(path):
    p=Path(path)
    if not p.is_file(): raise FileNotFoundError(p)
    if p.suffix.lower()=='.parquet': return pd.read_parquet(p)
    return pd.read_csv(p, sep='\t' if p.suffix.lower()=='.tsv' else ',')

def check_ids(df, id_col):
    if id_col not in df or df[id_col].isna().any() or df[id_col].duplicated().any():
        raise ValueError(f'{id_col}: 각 예측 단위에 결측 없는 고유 ID가 필요합니다.')

def split_rows(df, target=None, task='classification', strategy='random', group=None, time_col=None, fraction=.25, gap=0):
    idx=np.arange(len(df))
    if not 0<fraction<1: raise ValueError('validation fraction은 0~1 사이여야 합니다.')
    if strategy=='group':
        if not group or df[group].isna().any(): raise ValueError('유효한 그룹 열이 필요합니다.')
        a,b=next(GroupShuffleSplit(n_splits=1,test_size=fraction,random_state=SEED).split(df,groups=df[group]))
        assert set(df.iloc[a][group]).isdisjoint(set(df.iloc[b][group]))
    elif strategy=='time':
        if not time_col: raise ValueError('시간 열을 지정하세요.')
        t=pd.to_datetime(df[time_col],errors='raise')
        if t.isna().any(): raise ValueError('시간 결측을 해결하세요.')
        unique=np.sort(t.unique());cut=int(len(unique)*(1-fraction))
        if cut<=gap or cut>=len(unique): raise ValueError('시간 분할에 필요한 데이터가 부족합니다.')
        a=idx[t<unique[cut-gap]];b=idx[t>=unique[cut]]
        assert t.iloc[a].max()<t.iloc[b].min()
    elif strategy in ('random','stratified'):
        strat=df[target] if task=='classification' and target else None
        if strat is not None and strat.value_counts().min()<2:
            raise ValueError('표본 1개인 클래스가 있습니다. 병합/수집/분할 정책을 검토하세요.')
        a,b=train_test_split(idx,test_size=fraction,random_state=SEED,stratify=strat)
    else: raise ValueError(f'지원하지 않는 분할: {strategy}')
    if not len(a) or not len(b): raise ValueError('빈 학습/검증 분할')
    if task=='classification' and target and set(df.iloc[b][target])-set(df.iloc[a][target]):
        raise ValueError('학습에 없는 클래스가 검증에 존재합니다.')
    return np.asarray(a),np.asarray(b)

def clean_tabular(X):
    out=X.copy()
    for c in out:
        if pd.api.types.is_numeric_dtype(out[c]):
            out[c]=pd.to_numeric(out[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
        else: out[c]=out[c].map(lambda v: str(v) if pd.notna(v) else np.nan)
    return out

def tabular_preprocessor(X, robust=False):
    num=X.select_dtypes(include='number').columns.tolist()
    cat=[c for c in X if c not in num]
    parts=[]
    if num: parts.append(('num',make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),RobustScaler() if robust else StandardScaler()),num))
    if cat: parts.append(('cat',make_pipeline(SimpleImputer(strategy='constant',fill_value='__MISSING__',keep_empty_features=True),OneHotEncoder(handle_unknown='ignore',min_frequency=2)),cat))
    if not parts: raise ValueError('특징 열이 없습니다.')
    return ColumnTransformer(parts)

def metrics_for(model,X,y,task):
    pred=model.predict(X)
    if task=='regression': return {'mae':float(mean_absolute_error(y,pred)),'rmse':float(np.sqrt(mean_squared_error(y,pred))),'r2':float(r2_score(y,pred))}
    out={'accuracy':float(accuracy_score(y,pred)),'f1_macro':float(f1_score(y,pred,average='macro',zero_division=0))}
    if hasattr(model,'predict_proba'):
        p=model.predict_proba(X);classes=model.classes_
        out['log_loss']=float(log_loss(y,p,labels=classes))
        if len(classes)==2 and len(np.unique(y))==2: out['roc_auc']=float(roc_auc_score(np.asarray(y)==classes[1],p[:,1]))
    return out

def fit_compare(candidates,X,y,a,b,task,metric,budget=120):
    direction={'accuracy':True,'f1_macro':True,'roc_auc':True,'r2':True,'log_loss':False,'mae':False,'rmse':False}
    if metric not in direction: raise ValueError('지원 지표를 선택하거나 metrics_for를 확장하세요.')
    t0=time.monotonic();rows=[];fitted={}
    for name,estimator in candidates.items():
        if rows and time.monotonic()-t0>=budget: break
        start=time.monotonic();model=clone(estimator).fit(X.iloc[a] if hasattr(X,'iloc') else X[a],y.iloc[a] if hasattr(y,'iloc') else y[a])
        stats=metrics_for(model,X.iloc[b] if hasattr(X,'iloc') else X[b],y.iloc[b] if hasattr(y,'iloc') else y[b],task)
        if metric not in stats or not np.isfinite(stats[metric]): raise ValueError(f'{metric}: 이 분할/모델에서 계산할 수 없습니다.')
        rows.append({'model':name,**stats,'seconds':time.monotonic()-start});fitted[name]=model
    table=pd.DataFrame(rows).sort_values(metric,ascending=not direction[metric]);display(table)
    best=table.iloc[0]['model']
    return best,fitted[best],table

def write_submission(ids,pred,id_col,target_cols,output,sample_path=None,probabilities=False):
    ids=pd.Series(ids).reset_index(drop=True)
    arr=np.asarray(pred)
    if arr.ndim==1: arr=arr[:,None]
    if arr.ndim!=2 or arr.shape!=(len(ids),len(target_cols)): raise ValueError('예측의 행/열 수와 제출 규격이 다릅니다.')
    if ids.isna().any() or ids.duplicated().any(): raise ValueError('제출 ID 결측/중복')
    if len(set(target_cols))!=len(target_cols) or id_col in target_cols: raise ValueError('제출 열 이름 중복')
    out=pd.DataFrame(arr,columns=target_cols);out.insert(0,id_col,ids.to_numpy())
    if out.isna().any().any(): raise ValueError('제출 값 결측')
    numeric=out[target_cols].select_dtypes(include='number')
    if numeric.size and not np.isfinite(numeric.to_numpy()).all(): raise ValueError('제출 값 무한대')
    if probabilities:
        v=arr.astype(float)
        if not np.isfinite(v).all() or (v<0).any() or (v>1).any(): raise ValueError('확률 범위 오류')
        if v.shape[1]>1 and not np.allclose(v.sum(axis=1),1,atol=1e-5): raise ValueError('확률 합 오류')
    if sample_path:
        sample=read_table(sample_path);check_ids(sample,id_col)
        if set(sample.columns)!=set(out.columns): raise ValueError('sample_submission과 열이 다릅니다.')
        if len(sample)!=len(out) or set(sample[id_col])!=set(out[id_col]): raise ValueError('sample_submission과 ID가 다릅니다. ID 자료형도 확인하세요.')
        out=sample[[id_col]].merge(out,on=id_col,how='left',validate='one_to_one')[sample.columns]
    p=Path(output);p.parent.mkdir(parents=True,exist_ok=True);out.to_csv(p,index=False)
    display(out.head());print('저장:',p,'형태:',out.shape)
    return out

def classification_output(model,X,kind='label',order=None,positive=None):
    if kind=='label': return model.predict(X)
    if not hasattr(model,'predict_proba'): raise ValueError('확률을 지원하는 모델이 필요합니다.')
    classes=list(model.classes_);p=model.predict_proba(X)
    if kind=='positive_probability':
        if positive not in classes: raise ValueError('positive_class를 실제 클래스 값으로 지정하세요.')
        return p[:,classes.index(positive)]
    if kind!='probability' or order is None or len(order)!=len(classes) or set(order)!=set(classes):
        raise ValueError('공식 제출 열에 대응하는 class_order를 지정하세요.')
    return p[:,[classes.index(v) for v in order]]


## 데이터 준비

기본 예제에는 범주형과 결측이 포함됩니다.

In [ ]:
if DEMO:
    from sklearn.datasets import make_classification, make_regression
    if TASK=='classification': values,labels=make_classification(n_samples=260,n_features=6,n_informative=4,random_state=SEED)
    else: values,labels=make_regression(n_samples=260,n_features=6,noise=8,random_state=SEED)
    all_df=pd.DataFrame(values,columns=[f'x{i}' for i in range(6)])
    all_df['category']=np.where(all_df.x0>0,'A','B');all_df.loc[::13,'x1']=np.nan
    all_df[ID]=np.arange(len(all_df));all_df[TARGET]=labels
    train=all_df.iloc[:220].copy();test=all_df.iloc[220:].drop(columns=TARGET).copy()
else: train=read_table(TRAIN_PATH);test=read_table(TEST_PATH)
check_ids(train,ID);check_ids(test,ID)
if train[TARGET].isna().any(): raise ValueError('학습 정답 결측을 확인하세요.')
df=train


## 형식·수치형·상관 분석

ID와 정답의 상관은 모델 특징 선택에 그대로 사용하지 마세요.

In [ ]:
print('shape:',df.shape,'duplicates:',df.duplicated().sum())
display(pd.DataFrame({'dtype':df.dtypes.astype(str),'missing_ratio':df.isna().mean(),'unique':df.nunique(dropna=False)}))
display(df.head())
num=df.select_dtypes(include='number')
if not num.empty:
    display(num.describe().T)
    num.iloc[:,:8].hist(bins=20,figsize=(12,6));plt.tight_layout();plt.show()
    corr=num.iloc[:,:12].corr()
    fig,ax=plt.subplots(figsize=(7,5));im=ax.imshow(corr,vmin=-1,vmax=1,cmap='coolwarm')
    ax.set_xticks(range(len(corr)),corr.columns,rotation=90);ax.set_yticks(range(len(corr)),corr.columns);fig.colorbar(im);plt.show()


## 범주형·타깃·이상치 탐색

표본이 적은 범주와 사후에 생성된 누수 열을 확인합니다. 탐색만 하고 전체 데이터로 전처리를 학습하지 않습니다.

In [ ]:
for col in df.select_dtypes(exclude='number').columns[:8]: display(df[col].value_counts(dropna=False).head(15))
if TASK=='classification': display(df[TARGET].value_counts(normalize=True))
for col in df.select_dtypes(include='number').columns:
    if col in (ID,TARGET): continue
    q1,q3=df[col].quantile([.25,.75]);iqr=q3-q1
    print(col,'IQR outlier ratio:',((df[col]<q1-1.5*iqr)|(df[col]>q3+1.5*iqr)).mean())
if not DEMO:
    print('열 차이:',set(train.columns)-set(test.columns),set(test.columns)-set(train.columns))
